# Strategy Report Structure

This notebook contains the report outline for Task 1 and Task 2, following the assignment process and the requested structure.

## 1. Executive summary

The proposed strategy is a commodity trend-following system built with a CTA advisory-style structure. It aims to outperform an equal-weighted commodity benchmark by capturing trends and letting positions run until they show signs of deterioration. The design is layered, following CTA principles: Layer 1 identifies the macro regime, and Layer 2 defines entry timing. Rather than using a fixed time-window rebalancing approach, the strategy seeks to stay in a trend until it weakens, which requires a different position and money-management architecture. This leads to a bottom-up, risk-budgeting framework. The core idea is an "umbrella" allocation where capital for each commodity is managed in silos. The slot manager handles single-commodity trade execution, while the portfolio manager oversees overall exposure, weight optimization, and portfolio-level capital management.



- Briefly describe the strategy objective.
- State the overall approach and what the strategy aims to achieve.
- Mention the high-level conclusion or observed performance.

## 2. Strategy motivation

- Why commodity trend-following is a reasonable choice.
- Why a layered signal design is used.
- What the main trade-offs are and why the design is disciplined.

## 3. Overall architecture / pipeline

- Describe the end-to-end flow: data → indicators → signals → positions → portfolio → returns → analytics.
- Mention that Task 1 is implemented in `tasks/task_1/strategy.py`.
- Highlight the main code modules supporting the pipeline.

## 4. Task 1 — Base Strategy Structure

The Task 1 strategy is built as a sequential pipeline where each layer has a precise and limited role: regime identification, entry timing, position management, and portfolio sizing. The design deliberately separates these concerns so that no single component carries too much responsibility.

---

### Layer 1 — Trend Regime Filter

Layer 1 acts as a persistent gatekeeper. Its sole purpose is to determine whether a commodity is in a directional trending environment worthy of trading. It combines a classical moving-average crossover (SMA50 vs SMA200) with a slope confirmation on the slow SMA, measured over a 100-day lookback. Both conditions must agree simultaneously for a regime to be active — the signal is persistent and remains on every day the conditions hold, not only at the crossover date.

| Regime | Condition |
|--------|-----------|
| Long (+1) | SMA50 > SMA200 **and** SMA200 slope > 0 |
| Short (−1) | SMA50 < SMA200 **and** SMA200 slope < 0 |
| Neutral (0) | Conditions disagree |

---

### Layer 2 — Entry Timing

Layer 2 operates strictly inside an active Layer 1 regime. It identifies the precise moment to open a position by requiring price momentum confirmation via an EMA14 crossover and an RSI regime break. The RSI flag remains active for a configurable window after the break, filtering out EMA crosses that lack genuine momentum backing.

| Direction | Condition |
|-----------|-----------|
| Long (+1) | Layer 1 = +1, price crosses above EMA14, RSI recently crossed above long threshold |
| Short (−1) | Layer 1 = −1, price crosses below EMA14, RSI recently crossed below short threshold |

The signal is recorded at close of day *t*; the slot manager shifts execution to *t+1* to preserve the no-lookahead constraint.

---

### Position Management — Slot Manager

The slot manager converts entry events into actual positions, enforcing at most one open position per commodity at any time. Repeated signals while a position is already live are suppressed. Exits are triggered by the first of three conditions, whichever arrives first:

| Exit Trigger | Logic |
|--------------|-------|
| ATR stop | Price moves against the position by a fixed ATR multiple from entry |
| Bollinger Band profit exit | Price reaches the outer Bollinger Band in the direction of the trade |
| Regime exit | Layer 1 regime no longer matches the open position direction |

---

### Portfolio Sizing — Risk Budgeting

The portfolio manager translates open positions into weights using an ATR-based risk budgeting approach. Each position is sized so that the expected loss at the stop distance corresponds to a fixed fraction of capital. The formula is:

9601w_i = \min\left(rac{r}{d_i},\ w_{\max}
ight) 	imes 	ext{direction}_i9601

where:
- $ =  — fixed capital fraction risked per trade (e.g. 1%)
- $ =  — stop distance as a fraction of price
- {\max}$ = maximum weight cap per commodity

Weights are locked at entry and remain fixed until the position closes. New positions are skipped if opening them would breach the gross leverage cap. All weights are shifted by  before being applied to returns.

### 4.1 Task 1 performance

- Report core metrics: annual return, volatility, Sharpe, Sortino, Calmar, max drawdown, hit rate.
- Summarize the observed performance of Task 1.

### 4.2 Task 1 validation

- Explain the validation approach used for Task 1.
- Mention training/test split, no-lookahead conventions, and execution lag.
- Describe why the results are robust and not overfit.

## 5. Task 2 enhancements

- State that Task 2 preserves Task 1 identity.
- List the enhancement package: multi-lookback conviction, macro conviction modulation, stricter short RSI + execution-time cap, Parabolic SAR-style exit, transaction-cost-aware returns.
- Explain each enhancement briefly and why it was added.

### 5.1 Task 2 performance

- Report the same metrics as Task 1 for consistency.
- Include both gross and net performance, so the effect of transaction costs is visible.
- Compare Task 2 performance against Task 1 when possible.
- Highlight cost sensitivity findings, showing how net Sharpe and return change as cost assumptions vary.

### 5.2 Task 2 validation

- Explain the Task 2 validation procedure.
- Mention training diagnostics, parameter selection, and OOS or holdout evaluation.
- Describe evidence that the enhancements are robust.
- Discuss cost sensitivity analysis explicitly: how transaction costs were modeled and whether the strategy remains viable under realistic cost assumptions.

## 6. In-depth analysis: Task 2

- Deep dive into one or two major Task 2 enhancements.
- Explain why those changes matter.
- Use empirical evidence or diagnostics if available.

## 7. Conclusions

- Restate the main findings.
- Summarize the strategy strengths and risks.
- Optionally mention next-step ideas.